# Week 3 · Correlation Analysis & Multicollinearity + EDA Synthesis — Assignment
*DS 207 · Week 3*

*EC Corro, built in collaboration with Claude, ChatGPT, and Gemini.*

## Business Scenario Recap

Bluewater Resorts operates three sister properties across Cebu — Bluewater Maribago, Bluewater Mactan Ridge, and Bluewater Bantayan Cove. Marco Dela Peña, VP of Guest Experience and Revenue Strategy, is preparing the next fiscal year's staffing model, service-level targets, and amenities investment plan, and has asked for a rigorous pass through the resorts' daily operational data before any correlation-based claim goes into a board deck: which metrics genuinely move together, which apparent relationships are statistical illusions, and which candidate model inputs are dangerously redundant with each other.

## Submission Instructions

- File name: `ds207_03_correlation_multicollinearity_<lastname>.ipynb`
- Submit to your shared Google Drive folder.
- Due: 11:59 PM the day after the associated session.

## AI Use Policy

Generative AI tools are permitted for concept clarification, pseudocode, debugging, and idea generation. You must be able to explain any submitted answer without AI assistance, and you must cite any AI tool use (tool name and how it was used) in the relevant cell or a footnote.

## Points

Section 1 (Concepts, Computation & Coding): 30 pts (3 items × 10 pts)
Section 2 (Business Context & Reasoning): 30 pts (2 items × 15 pts)
**Total: 60 pts**

## Setup

Run the cell below first. It generates this week's dataset — 450 daily operational snapshots pooled across the three Bluewater properties — seeded for reproducibility. Do not modify this cell.

In [19]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
rng = np.random.default_rng(207)

N = 450
resorts = rng.choice(
    ["Bluewater Maribago", "Bluewater Mactan Ridge", "Bluewater Bantayan Cove"],
    size=N, p=[0.4, 0.35, 0.25]
)
resort_tier = np.where(resorts == "Bluewater Mactan Ridge", 1.15,
                np.where(resorts == "Bluewater Maribago", 1.0, 0.85))

dates = pd.date_range("2026-01-05", periods=200, freq="D")
date = rng.choice(dates, size=N)

guest_count = np.clip(rng.normal(120, 35, N) * resort_tier, 25, None).round().astype(int)
room_rate_php = np.clip(rng.normal(6800, 1400, N) * resort_tier + rng.normal(0, 300, N), 2500, None).round(-1)
total_staff_count = np.clip(0.28 * guest_count + rng.normal(0, 4, N) + 8, 10, None).round().astype(int)
housekeeping_staff_count = np.clip(0.55 * total_staff_count + rng.normal(0, 1.8, N), 3, None).round().astype(int)
rainfall_mm = np.clip(rng.gamma(1.6, 9.0, N), 0, 95).round(1)
length_of_stay_nights = np.clip(rng.gamma(3.0, 1.15, N), 1, 14).round(1)

staff_to_guest_ratio = (total_staff_count / guest_count).round(4)

review_response_time_hours = np.clip(
    rng.normal(14, 7, N) - 1.8 * (staff_to_guest_ratio - staff_to_guest_ratio.mean()) * 10, 0.5, 60
).round(1)

spa_bookings_per_100_guests = np.clip(
    4.5 * length_of_stay_nights + rng.normal(0, 6, N), 0, None
).round(1)

satisfaction_raw = (
    5.55
    + 9.5 * (staff_to_guest_ratio - staff_to_guest_ratio.min())
    - 1.35 * np.log1p(review_response_time_hours)
    + 0.42 * length_of_stay_nights
    + rng.normal(0, 0.55, N)
)
guest_satisfaction_score = np.clip(satisfaction_raw, 1, 10).round(2)

occupancy_raw = (
    0.82
    - 0.0035 * rainfall_mm
    - 0.000012 * (room_rate_php - room_rate_php.mean())
    + rng.normal(0, 0.05, N)
)
occupancy_rate = np.clip(occupancy_raw, 0.15, 0.99).round(3)

bluewater = pd.DataFrame({
    "date": date, "resort": resorts, "guest_count": guest_count,
    "room_rate_php": room_rate_php, "total_staff_count": total_staff_count,
    "housekeeping_staff_count": housekeeping_staff_count,
    "staff_to_guest_ratio": staff_to_guest_ratio,
    "review_response_time_hours": review_response_time_hours,
    "rainfall_mm": rainfall_mm, "length_of_stay_nights": length_of_stay_nights,
    "spa_bookings_per_100_guests": spa_bookings_per_100_guests,
    "guest_satisfaction_score": guest_satisfaction_score,
    "occupancy_rate": occupancy_rate,
}).sort_values("date").reset_index(drop=True)

miss_mask = rng.random(len(bluewater)) < (0.05 + 0.15 * (bluewater["rainfall_mm"] > bluewater["rainfall_mm"].quantile(0.75)))
bluewater.loc[miss_mask, "review_response_time_hours"] = np.nan

outlier_idx = rng.choice(bluewater.index, size=6, replace=False)
bluewater.loc[outlier_idx[:2], "room_rate_php"] = bluewater.loc[outlier_idx[:2], "room_rate_php"] * 4.2
bluewater.loc[outlier_idx[2:4], "guest_satisfaction_score"] = 0.4
bluewater.loc[outlier_idx[4:6], "total_staff_count"] = bluewater.loc[outlier_idx[4:6], "total_staff_count"] + 55

clean = bluewater.dropna(subset=["review_response_time_hours"]).copy()
print(f"Raw export: {bluewater.shape[0]} rows.  Complete-case frame `clean`: {clean.shape[0]} rows.")
bluewater.head()

Raw export: 450 rows.  Complete-case frame `clean`: 402 rows.


,date,resort,guest_count,room_rate_php,total_staff_count,housekeeping_staff_count,staff_to_guest_ratio,review_response_time_hours,rainfall_mm,length_of_stay_nights,spa_bookings_per_100_guests,guest_satisfaction_score,occupancy_rate
0,2026-01-05,Bluewater Mactan Ridge,87,12560.0,32,18,0.3678,8.2,13.0,2.5,18.3,4.44,0.706
1,2026-01-05,Bluewater Mactan Ridge,229,8750.0,76,40,0.3319,10.1,6.9,4.8,25.4,4.79,0.802
2,2026-01-05,Bluewater Maribago,172,6490.0,62,31,0.3605,27.0,1.5,4.3,10.6,3.82,0.835
3,2026-01-06,Bluewater Maribago,112,6260.0,35,16,0.3125,17.2,18.7,1.9,10.1,2.64,0.849
4,2026-01-06,Bluewater Mactan Ridge,143,6990.0,43,26,0.3007,17.0,11.7,4.8,28.2,5.24,0.759


## Section 1: Concepts, Computation & Coding

### Item 1 (10 pts) — Covariance and Pearson Correlation

Marco wants to know whether the staff-to-guest ratio is worth protecting in next year's budget as a genuine driver of guest satisfaction.

**(a) Concept (3 pts).** Covariance between `staff_to_guest_ratio` and `guest_satisfaction_score` is a small positive number (around 0.02), while the Pearson correlation is a much larger number (around 0.34) for the exact same relationship. Explain why these two numbers look so different in magnitude even though they describe the same underlying relationship, and explain what rescaling covariance into a correlation coefficient buys you when comparing it to *other* relationships in this dataset (for example, `review_response_time_hours` vs. `guest_satisfaction_score`).

**Your answer:** Although covariance and correlation describe the same underlying relationship, these two concepts differ in scale because of how they are defined. Covariance depends on the units of the variables whereas correlation rescales covariance. If we check the values of the staff-to-guest-ratio, the values are mostly small decimal value, while the guest_satisfaction_score is on a scale basis. So if we multiply small decimal values (staff-to-guest-ratio) to a bigger number (guest_satisfaction_score) that will give us a small number. If we look at the formula of the Pearson correlation, it basically removes the units. 

Now, the benefit of rescaling the covariance into a correlation coefficient allows us to compare different variable pairs in the dataset, such as staffing ratios and satisfaction versus review response times and satisfaction without being misled by differences in measurement units or variable scales. 

**(b) By hand (4 pts).** Using the five-day extract below, compute the sample covariance and the Pearson correlation coefficient between `staff_to_guest_ratio` ($x$) and `guest_satisfaction_score` ($y$) by hand. Show every intermediate quantity ($\bar{x}$, $\bar{y}$, each deviation, each squared deviation, each cross-product, and the sums) before stating your final $r$.

| Day | $x$ (staff-to-guest ratio) | $y$ (satisfaction) |
|---|---|---|
| 1 | 0.4868 | 7.09 |
| 2 | 0.3578 | 3.87 |
| 3 | 0.3446 | 2.57 |
| 4 | 0.3364 | 3.10 |
| 5 | 0.3578 | 3.59 |

**Your work:**
$\bar{x} = 0.37668$, $\bar{y} = 4.044$

| Day | $x_i-\bar{x}$ | $y_i-\bar{y}$ | $(x_i-\bar{x})(y_i-\bar{y})$ | $(x_i-\bar{x})^2$ | $(y_i-\bar{y})^2$ |
|---|---|---|---|---|---|
| 1 | 0.11012 | 	3.046 |	0.3354 |	0.0121 |	9.2781 |
| 2 | -0.01888 | 	-0.174 |	0.0033 |	0.0004 |	0.0303 |
| 3 | -0.03208 | 	-1.474 |	0.0473 |	0.0010 |	2.1727 |
| 4 | -0.04028 | 	-0.944 |	0.0380 |	0.0016 |	0.8911 |
| 5 | -0.01888 | 	-0.454 |	0.0086 |	0.0004 |	0.2061 |
| **Sum** | | | **0.4326** | **0.0155** | **12.5783** |

$$
\text{Cov}(X,Y) = \frac{0.4326}{5-1} = 0.1081, \qquad
s_X = \sqrt{\frac{0.0155}{4}} = 0.0623, \qquad
s_Y = \sqrt{\frac{12.5783}{4}} = 1.774
$$

$$
r = \frac{0.1081}{0.0623 \times 1.774} \approx 0.98
$$

**(c) By tool (3 pts).** Complete the cell below to verify your by-hand answer, then compute the same statistics on the full `clean` analysis frame and report whether the full-sample result supports the same conclusion as your five-day extract.

In [25]:
# TODO: define x and y as the five values from the table above

x_sample = np.array([0.4868, 0.3578, 0.3446, 0.3364, 0.3578]) ###staff-to-guest-ratio
y_sample = np.array([7.09, 3.87, 2.57, 3.10, 3.59]) ####satisfaction

# TODO: compute covariance and Pearson r using numpy/scipy and print them
# (they should match your by-hand answer in part b)
xbar, ybar = x_sample.mean(), y_sample.mean()
cov_manual = np.sum((x_sample- xbar) * (y_sample - ybar)) / (len(x_sample) - 1)
sx_manual = np.sqrt(np.sum((x_sample - xbar) ** 2) / (len(x_sample) - 1))
sy_manual = np.sqrt(np.sum((y_sample - ybar) ** 2) / (len(x_sample) - 1))
r_manual = cov_manual / (sx_manual * sy_manual)

print(f"xbar = {xbar:.4f}, ybar = {ybar:.4f}")
print(f"Manual covariance = {cov_manual:.5f}")
print(f"Manual r = {r_manual:.4f}")

r_scipy, p_scipy = stats.pearsonr(x_sample, y_sample)
print(f"scipy.stats.pearsonr r = {r_scipy:.4f}, p = {p_scipy:.4f}  (matches by-hand result)")


# TODO: compute covariance and Pearson r (with p-value) on the full `clean` DataFrame
# for staff_to_guest_ratio vs. guest_satisfaction_score, and print n, r, and p
print("\nCovariance and Pearson r of the Staff-to-Guests-ratio and Guest Satisfaction Score ")
r_full, p_full = stats.pearsonr(clean["staff_to_guest_ratio"], clean["guest_satisfaction_score"])
cov_full = np.cov(clean["staff_to_guest_ratio"], clean["guest_satisfaction_score"], ddof=1)[0, 1]
print(f"Full sample (n={len(clean)}): covariance = {cov_full:.5f}, Pearson r = {r_full:.4f}, p = {p_full:.2e}")


xbar = 0.3767, ybar = 4.0440
Manual covariance = 0.10815
Manual r = 0.9800
scipy.stats.pearsonr r = 0.9800, p = 0.0034  (matches by-hand result)

Covariance and Pearson r of the Staff-to-Guests-ratio and Guest Satisfaction Score 
Full sample (n=402): covariance = 0.02493, Pearson r = 0.3384, p = 3.18e-12


### Item 2 (10 pts) — Spearman Correlation

Marco's operations team wants to set a review-response-time service-level target, but they are worried the relationship with satisfaction might not be a straight line.

**(a) Concept (3 pts).** Explain, in terms of what each coefficient actually measures, why Spearman's rank correlation is a more appropriate first check than Pearson's correlation for the claim "slower review response times are associated with lower satisfaction," even before you have plotted the data. Your answer should reference what "monotonic" means and why it is a weaker (easier to satisfy) requirement than "linear."

**Your answer:** Spearman’s rank correlation measures whether two variables have a monotonic relationship. A monotonic relationship means that as one variable increases, the other consistently increases or decreases, even if the rate of change is uneven. This makes Spearman flexible because it does not require the data to follow a straight line.

Pearson’s correlation, by contrast, measures the strength of a linear relationship. It assumes that the data points align roughly along a straight line with a constant slope. This is a stricter condition, and if the relationship is curved or irregular, Pearson may underestimate the strength of the association.

In the context of the claim “slower review response times are associated with lower satisfaction,” Spearman asks only whether longer response times consistently correspond to lower satisfaction scores. It does not matter whether the decline is sharp at first and then levels off, or whether it follows another curved pattern. As long as the trend is consistently downward, Spearman will capture it, making it the more appropriate first check before plotting the data.

**(b) By hand (4 pts).** Using the five-day extract below, compute the Spearman rank correlation between `review_response_time_hours` ($x$) and `guest_satisfaction_score` ($y$) by hand. Show the ranks you assign to each variable, each rank difference $d_i$, each $d_i^2$, and the final $\rho$.

| Day | $x$ (response hrs) | $y$ (satisfaction) | rank($x$) | rank($y$) | $d$ | $d^2$ |
|---|---|---|---|---|---|---|
| 1 | 21.7 | 4.23 | 5 | 1 | 4 | 16 |
| 2 | 2.1 | 7.70 | 1 | 5 | -4 | 16 |
| 3 | 10.0 | 4.82 | 3 | 2 | 1 | 1 |
| 4 | 6.9 | 7.19 | 2 | 3 | -1 | 1 |
| 5 | 12.5 | 7.68 | 4 | 4 | 0 | 0 |
| | | | | | **Sum** | **34** |

$$
\rho = 1 - \frac{6 \times 34}{5(25-1)} = 1 - \frac{204}{120} = -0.7
$$


**Your work:**

**(c) By tool (3 pts).** Complete the cell below to verify your by-hand answer, then compute both Pearson $r$ and Spearman $\rho$ on the full `clean` analysis frame. Report both values and state whether the gap between them (if any) changes your interpretation from part (a).

In [30]:
# TODO: define x and y as the five values from the table above
x_sample2 = np.array([21.7, 2.1, 10.0, 6.9, 12.5])
y_sample2 = np.array([4.23, 7.70, 4.82, 7.19, 7.68])

# TODO: compute Spearman rho using scipy.stats.rankdata and/or scipy.stats.spearmanr
# and confirm it matches your by-hand answer
rank_x = stats.rankdata(x_sample2)
rank_y = stats.rankdata(y_sample2)
d = rank_x - rank_y
n = len(x_sample2)
rho_manual = 1 - (6 * np.sum(d ** 2)) / (n * (n ** 2 - 1))

print("x (response hrs):", x_sample2.tolist())
print("y (satisfaction):", y_sample2.tolist())
print("rank(x):", rank_x.tolist(), " rank(y):", rank_y.tolist())
print(f"sum(d^2) = {np.sum(d**2):.0f}")
print(f"Manual Spearman rho = {rho_manual:.4f}")

rho_scipy, p_scipy = stats.spearmanr(x_sample2, y_sample2)
r_pearson_5, _ = stats.pearsonr(x_sample2, y_sample2)
print(f"scipy.stats.spearmanr rho = {rho_scipy:.4f}  (matches by-hand result)")
print(f"For contrast, Pearson r on these same 5 points = {r_pearson_5:.4f}")

# TODO: on the full `clean` DataFrame, compute BOTH Pearson r and Spearman rho
# for review_response_time_hours vs. guest_satisfaction_score, with p-values, and print them
print(f"Using the complete-case analysis frame `clean` (n = {len(clean)}); "
      f"{bluewater['review_response_time_hours'].isna().sum()} rows with missing response time excluded "
      f"(missingness mechanism is diagnosed properly in the EDA Synthesis section below).")

r_full, p_r = stats.pearsonr(clean["review_response_time_hours"], clean["guest_satisfaction_score"])
rho_full, p_rho = stats.spearmanr(clean["review_response_time_hours"], clean["guest_satisfaction_score"])
r_log, _ = stats.pearsonr(np.log1p(clean["review_response_time_hours"]), clean["guest_satisfaction_score"])

print(f"\nFull sample (n={len(clean)}):")
print(f"  Pearson r (raw hours)          = {r_full:.4f}")
print(f"  Spearman rho                   = {rho_full:.4f}")
print(f"  Pearson r (on log1p(hours))    = {r_log:.4f}")

x (response hrs): [21.7, 2.1, 10.0, 6.9, 12.5]
y (satisfaction): [4.23, 7.7, 4.82, 7.19, 7.68]
rank(x): [5.0, 1.0, 3.0, 2.0, 4.0]  rank(y): [1.0, 5.0, 2.0, 3.0, 4.0]
sum(d^2) = 34
Manual Spearman rho = -0.7000
scipy.stats.spearmanr rho = -0.7000  (matches by-hand result)
For contrast, Pearson r on these same 5 points = -0.7118
Using the complete-case analysis frame `clean` (n = 402); 48 rows with missing response time excluded (missingness mechanism is diagnosed properly in the EDA Synthesis section below).

Full sample (n=402):
  Pearson r (raw hours)          = -0.5880
  Spearman rho                   = -0.5804
  Pearson r (on log1p(hours))    = -0.6266


### Item 3 (10 pts) — Correlation Matrices and Multicollinearity

Marco's analytics team is about to build a small internal model of guest satisfaction using every staffing and operational variable they have, including both `total_staff_count` and `housekeeping_staff_count` as separate inputs.

**(a) Concept (3 pts).** Explain why two variables can each show only a weak correlation with an outcome (guest satisfaction) and yet still cause serious problems when included together as predictors in the same regression model. Your answer should explain what the Variance Inflation Factor (VIF) actually quantifies.

**Your answer:** Two variables can each show only a weak correlation with guest satisfaction, but when you put them together in a regression model, the problem isn’t their link to the outcome, the problem is how they overlap with each other. If predictors share too much information, the model struggles to separate their individual effects, which makes the coefficients unstable, sometimes even flipping signs or showing inflated p‑values. This is exactly what the Variance Inflation Factor (VIF) is designed to measure. VIF tells you how much the variance of a regression coefficient is increased because of multicollinearity, essentially checking how well one predictor can be explained by the others. A VIF of 1 means no overlap, while higher values (commonly above 5 or 10) signal serious redundancy. In other words, even weak outcome correlations can hide strong inter‑predictor correlations, and VIF is the tool that reveals how that hidden redundancy undermines the reliability of your regression model.

**(b) By hand (4 pts).** Suppose regressing `total_staff_count` on all seven of the other candidate predictor variables yields an $R^2_j = 0.823$ for that regression. Using the VIF formula, compute $\text{VIF}_{\text{total\_staff\_count}}$ by hand, and state whether this value would typically be flagged under the common rule-of-thumb threshold discussed in lecture.

**Your work:**
$$
\text{VIF}_{\text{total\_staff}} = \frac{1}{1-0.823} = \frac{1}{0.177} \approx 5.65
$$

**(c) By tool (3 pts).** Complete the cell below to: (i) compute the full VIF table for the eight candidate predictor variables used in lecture, confirming your by-hand answer in part (b); and (ii) fit two OLS models predicting `guest_satisfaction_score` — one using only `length_of_stay_nights`, and a second adding `spa_bookings_per_100_guests` — and report what happens to the coefficient and standard error on `length_of_stay_nights` between the two models.

In [36]:
predictor_cols = ["room_rate_php", "total_staff_count", "housekeeping_staff_count",
                   "staff_to_guest_ratio", "review_response_time_hours", "rainfall_mm",
                   "length_of_stay_nights", "spa_bookings_per_100_guests"]

# TODO: build a standardized predictor matrix with a constant column, then compute
# the VIF for every predictor using variance_inflation_factor(); print the results
# sorted from highest to lowest VIF
X = clean[predictor_cols].copy()
X = (X - X.mean()) / X.std()          
X_with_const = sm.add_constant(X)

vif_table = pd.DataFrame({
    "feature": X_with_const.columns,
    "VIF": [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
})
vif_table = vif_table[vif_table["feature"] != "const"].sort_values("VIF", ascending=False).reset_index(drop=True)
print(vif_table.round(2).to_string(index=False))

# TODO: fit Model A: guest_satisfaction_score ~ length_of_stay_nights (statsmodels OLS)
# TODO: fit Model B: guest_satisfaction_score ~ length_of_stay_nights + spa_bookings_per_100_guests
# TODO: print the coefficient, standard error, and p-value for length_of_stay_nights in BOTH
# models, and compute how many times larger the standard error became in Model B
print("\n")
model_a = sm.OLS(clean["guest_satisfaction_score"],
                  sm.add_constant(clean[["length_of_stay_nights"]])).fit()
model_b = sm.OLS(clean["guest_satisfaction_score"],
                  sm.add_constant(clean[["length_of_stay_nights", "spa_bookings_per_100_guests"]])).fit()

print("Model A: satisfaction ~ length_of_stay_nights only")
print(f"  coef(length_of_stay_nights) = {model_a.params['length_of_stay_nights']:.5f}, "
      f"SE = {model_a.bse['length_of_stay_nights']:.5f}, p = {model_a.pvalues['length_of_stay_nights']:.3f}")

print("\nModel B: satisfaction ~ length_of_stay_nights + spa_bookings_per_100_guests")
print(f"  coef(length_of_stay_nights)       = {model_b.params['length_of_stay_nights']:.5f}, "
      f"SE = {model_b.bse['length_of_stay_nights']:.5f}, p = {model_b.pvalues['length_of_stay_nights']:.3f}")
print(f"  coef(spa_bookings_per_100_guests) = {model_b.params['spa_bookings_per_100_guests']:.5f}, "
      f"SE = {model_b.bse['spa_bookings_per_100_guests']:.5f}, p = {model_b.pvalues['spa_bookings_per_100_guests']:.3f}")


                    feature  VIF
          total_staff_count 5.65
   housekeeping_staff_count 5.65
      length_of_stay_nights 3.96
spa_bookings_per_100_guests 3.94
       staff_to_guest_ratio 1.06
                rainfall_mm 1.04
 review_response_time_hours 1.03
              room_rate_php 1.03


Model A: satisfaction ~ length_of_stay_nights only
  coef(length_of_stay_nights) = 0.40314, SE = 0.02879, p = 0.000

Model B: satisfaction ~ length_of_stay_nights + spa_bookings_per_100_guests
  coef(length_of_stay_nights)       = 0.40082, SE = 0.05680, p = 0.000
  coef(spa_bookings_per_100_guests) = 0.00051, SE = 0.01082, p = 0.962


## Section 2: Business Context & Reasoning

### Item 1 (15 pts) — Is the Spa Really Driving Satisfaction?

A member of Marco's team has pulled the correlation matrix and found that `spa_bookings_per_100_guests` correlates with `guest_satisfaction_score` at roughly $r = 0.50$ — noticeably stronger than most other pairs in the dataset. She is drafting a capital-expenditure proposal to expand spa capacity at all three properties, on the grounds that more spa bookings will directly raise guest satisfaction scores.

Using the `clean` analysis frame:

**Technical soundness (8 pts).** State an assumption about a plausible third variable that could be driving both spa bookings and satisfaction independently, without spa bookings causing satisfaction at all. Then use an appropriate technique to test whether the raw spa-bookings/satisfaction correlation survives once you control for that third variable. Show your computation and report the result clearly.

**Business reasoning (7 pts).** Based on your technical result, write a short recommendation (roughly 100–150 words) to Marco evaluating the spa capex proposal. Your recommendation should state clearly whether the proposal's underlying premise is supported by the data, and — if not — what the data suggests would be a more defensible lever to pursue instead.

In [40]:
# TODO: state your assumed confounding variable as a Python comment, e.g.:
# Given the scenario, we assume that the length_of_stay_nights variable is the confounding variable. 
# Guests who stay longer naturally have more opportunities to book spa treatments, 
# and they also tend to report higher satisfaction for reasons unrelated to spa use.

# TODO: compute the raw Pearson correlation between spa_bookings_per_100_guests and
# guest_satisfaction_score on the `clean` frame
r_with, p_with = stats.pearsonr(bluewater["spa_bookings_per_100_guests"], bluewater["guest_satisfaction_score"])

rate_q99 = bluewater["spa_bookings_per_100_guests"].quantile(0.99)
flagged = bluewater["spa_bookings_per_100_guests"] > rate_q99
clean_subset = bluewater.loc[~flagged]
r_without, p_without = stats.pearsonr(clean_subset["spa_bookings_per_100_guests"], clean_subset["guest_satisfaction_score"])

# TODO: compute the partial correlation between spa_bookings_per_100_guests and
# guest_satisfaction_score, controlling for your chosen third variable
# (you may write your own partial-correlation function, following the formula from lecture)
r_xy, p_xy = stats.pearsonr(clean["spa_bookings_per_100_guests"], clean["guest_satisfaction_score"])
r_xz, _ = stats.pearsonr(clean["spa_bookings_per_100_guests"], clean["length_of_stay_nights"])
r_yz, _ = stats.pearsonr(clean["guest_satisfaction_score"], clean["length_of_stay_nights"])

def partial_corr(rxy, rxz, ryz):
    return (rxy - rxz * ryz) / np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))

r_partial = partial_corr(r_xy, r_xz, r_yz)


# TODO: print both the raw and the partial correlation so they can be compared directly
print(f"Pearson r WITH the flagged high-rate entries    : {r_with:.4f}  (n={len(bluewater)})")
print(f"Pearson r WITH those {flagged.sum()} entries removed        : {r_without:.4f}  (n={len(clean_subset)})")

print(f"raw r(spa_bookings, satisfaction)              = {r_xy:.4f}  (p = {p_xy:.2e})  <- looks like a strong driver")
print(f"r(spa_bookings, length_of_stay)                 = {r_xz:.4f}")
print(f"r(satisfaction, length_of_stay)                 = {r_yz:.4f}")
print(f"partial r(spa_bookings, satisfaction | length)  = {r_partial:.4f}  <- once length of stay is held fixed, the link nearly vanishes")


Pearson r WITH the flagged high-rate entries    : 0.4783  (n=450)
Pearson r WITH those 5 entries removed        : 0.4243  (n=445)
raw r(spa_bookings, satisfaction)              = 0.4951  (p = 2.98e-26)  <- looks like a strong driver
r(spa_bookings, length_of_stay)                 = 0.8616
r(satisfaction, length_of_stay)                 = 0.5735
partial r(spa_bookings, satisfaction | length)  = 0.0024  <- once length of stay is held fixed, the link nearly vanishes


**Your recommendation to Marco:**

The data initially suggests a moderate link between spa bookings and guest satisfaction, but once we account for length of stay, that connection weakens considerably given that we have a partial r of 0.0024. Guests who stay longer naturally have more chances to use the spa and also tend to report higher satisfaction for reasons beyond spa use. This overlap makes it look like spa activity is driving satisfaction when, in reality, the true driver is how long guests remain at the property. Because the adjusted analysis shows the spa effect nearly disappears after controlling for stay duration, the proposal to expand spa capacity isn’t supported by the evidence. A stronger strategy would be to focus on encouraging longer visits, let's say through bundled packages, loyalty rewards, or extended‑stay offers, since duration is what consistently boosts both spa usage and satisfaction. This approach is more likely to deliver meaningful improvements in guest experience and overall returns.

### Item 2 (15 pts) — Choosing a Staffing Metric for the Board Deck

Marco's team currently tracks both `total_staff_count` and `housekeeping_staff_count` as separate KPIs in the monthly operations report, and both are candidate inputs for a planned model of `occupancy_rate`. Before that model is built, Marco wants a recommendation on whether both staffing metrics should be kept as separate model inputs going forward.

Using the `clean` analysis frame:

**Technical soundness (8 pts).** Quantify the relationship between `total_staff_count` and `housekeeping_staff_count`, and then fit two OLS models predicting `occupancy_rate` — one with `total_staff_count` alone, and a second adding `housekeeping_staff_count` — to demonstrate what happens to the `total_staff_count` coefficient and its standard error when the second variable is added.

**Business reasoning (7 pts).** Based on your technical result, write a short recommendation (roughly 100–150 words) to Marco on how the two staffing metrics should be handled in future reporting and modeling. Your recommendation should name which metric (or combination) you would keep as the standard KPI, and explain the operational risk of continuing to report and model both as if they were independent.

In [46]:
# TODO: compute the Pearson correlation between total_staff_count and housekeeping_staff_count
r_full, p_full = stats.pearsonr(clean["total_staff_count"], clean["housekeeping_staff_count"])
cov_full = np.cov(clean["total_staff_count"], clean["housekeeping_staff_count"], ddof=1)[0, 1]
print(f"Full sample (n={len(clean)}): covariance = {cov_full:.5f}, Pearson r = {r_full:.4f}, p = {p_full:.2e}")

# TODO: fit Model A: occupancy_rate ~ total_staff_count (statsmodels OLS)
# TODO: fit Model B: occupancy_rate ~ total_staff_count + housekeeping_staff_count
# TODO: print the coefficient, standard error, and p-value for total_staff_count in BOTH
# models, and compute how many times larger the standard error became in Model B
print("\n")
model_a = sm.OLS(clean["occupancy_rate"],
                  sm.add_constant(clean[["total_staff_count"]])).fit()
model_b = sm.OLS(clean["occupancy_rate"],
                  sm.add_constant(clean[["total_staff_count", "housekeeping_staff_count"]])).fit()

print("Model A: occupancy_rate ~ total_staff_count only")
print(f"  coef(total_staff_count) = {model_a.params['total_staff_count']:.5f}, "
      f"SE = {model_a.bse['total_staff_count']:.5f}, p = {model_a.pvalues['total_staff_count']:.3f}")

print("\nModel B: occupancy_rate ~ total_staff_count + housekeeping_staff_count")
print(f"  coef(total_staff_count)       = {model_b.params['total_staff_count']:.5f}, "
      f"SE = {model_b.bse['total_staff_count']:.5f}, p = {model_b.pvalues['total_staff_count']:.3f}")
print(f"  coef(housekeeping_staff_count) = {model_b.params['housekeeping_staff_count']:.5f}, "
      f"SE = {model_b.bse['housekeeping_staff_count']:.5f}, p = {model_b.pvalues['housekeeping_staff_count']:.3f}")

se_inflation = model_b.bse["total_staff_count"] / model_a.bse["total_staff_count"]
print(f"\nStandard error on total_staff_count grew {se_inflation:.2f}x just from adding its near-duplicate.")


Full sample (n=402): covariance = 71.00633, Pearson r = 0.9047, p = 2.84e-150


Model A: occupancy_rate ~ total_staff_count only
  coef(total_staff_count) = -0.00025, SE = 0.00030, p = 0.417

Model B: occupancy_rate ~ total_staff_count + housekeeping_staff_count
  coef(total_staff_count)       = 0.00090, SE = 0.00071, p = 0.204
  coef(housekeeping_staff_count) = -0.00229, SE = 0.00128, p = 0.074

Standard error on total_staff_count grew 2.34x just from adding its near-duplicate.


**Your recommendation to Marco:**
The analysis shows that total staff count and housekeeping staff count overlap heavily, which makes them poor candidates to treat as independent predictors. When both are included in models, the redundancy inflates uncertainty and causes unstable coefficients. For reporting and modeling going forward, I recommend using total staff count as the standard KPI. It gives a clear picture of overall staffing levels without duplicating information. Housekeeping staff can still be tracked internally for operational purposes, but it shouldn’t be modeled as an independent driver. If both metrics continue to be reported side by side, managers risk drawing misleading conclusions, thinking one metric is more influential than it really is. Consolidating around a single KPI will make reporting cleaner and ensure that modeling results are more reliable and easier to act on.

## References

All external sources used in this assignment — textbooks, papers, articles, online references, code snippets adapted from elsewhere, and any generative AI tool use — must be cited per course policy. Cite AI tool use specifically: name the tool, and briefly describe what it was used for (for example, concept clarification, debugging, or idea generation). Reminder: you must be able to explain any submitted answer without AI assistance.

- Week 3 Lecture Notes 
- Microsoft Copilot - used for idea generation and debugging